In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from phase1_dann import (
    EfficientNetDANN,
    Phase1Config,
    extract_features,
    load_origa_dataframe,
    load_rop_dataframe,
    run_tsne,
    split_rop_by_patient,
    train_phase1_kfold,
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device =', device)

In [ ]:
# Ajuste os caminhos se necessário
cfg = Phase1Config(
    origa_csv='/backup/lucas/datasets/origa/ORIGA/OrigaList.csv',
    origa_images_dir='/backup/lucas/datasets/origa/ORIGA/Images',
    rop_images_dir='/backup/pedro_fonseca/PATIENT_ROP/DATASET/images_stack/images_stack',
    rop_metadata_csv=None,
    output_dir='dann-rop-v3/artifacts',
)

source_df = load_origa_dataframe(cfg.origa_csv, cfg.origa_images_dir)
rop_df = load_rop_dataframe(cfg.rop_images_dir, cfg.rop_metadata_csv)
rop_train_df, rop_test_df = split_rop_by_patient(rop_df, cfg.rop_test_size, cfg.seed)

split_dir = Path(cfg.output_dir) / 'splits'
split_dir.mkdir(parents=True, exist_ok=True)
source_df.to_csv(split_dir / 'origa_full.csv', index=False)
rop_train_df.to_csv(split_dir / 'rop_train_split.csv', index=False)
rop_test_df.to_csv(split_dir / 'rop_test_split.csv', index=False)

meta = {
    'origa_images': int(len(source_df)),
    'rop_train_images': int(len(rop_train_df)),
    'rop_test_images': int(len(rop_test_df)),
    'rop_train_patients': int(rop_train_df['patient_id'].nunique()),
    'rop_test_patients': int(rop_test_df['patient_id'].nunique()),
}
with open(split_dir / 'split_meta.json', 'w', encoding='utf-8') as f:
    json.dump(meta, f, indent=2)

meta

In [ ]:
# Opcional: treinar Fase 1 aqui
RUN_PHASE1 = False

if RUN_PHASE1:
    results = train_phase1_kfold(source_df=source_df, target_df=rop_train_df, cfg=cfg, device=device)
    print('Treino finalizado. mean_auc=', results['mean_auc'])
else:
    print('Treino pulado (RUN_PHASE1=False).')

In [ ]:
def sample_df(df, n=800, seed=42):
    n = min(n, len(df))
    return df.sample(n=n, random_state=seed).reset_index(drop=True)

def make_tsne_plot(emb, meta, title):
    plot_df = meta.copy()
    plot_df['tsne_1'] = emb[:, 0]
    plot_df['tsne_2'] = emb[:, 1]
    plot_df['label_name'] = plot_df['label'].map({0: 'Normal', 1: 'Doença'})

    color_map = {'ORIGA': '#1f77b4', 'ROP': '#ff7f0e'}
    marker_map = {'Normal': 'o', 'Doença': 'x'}

    fig, ax = plt.subplots(figsize=(10, 8))
    for domain in sorted(plot_df['domain'].unique()):
        for label_name in ['Normal', 'Doença']:
            sub = plot_df[(plot_df['domain'] == domain) & (plot_df['label_name'] == label_name)]
            if len(sub) == 0:
                continue
            ax.scatter(
                sub['tsne_1'],
                sub['tsne_2'],
                c=color_map.get(domain, '#333333'),
                marker=marker_map.get(label_name, 'o'),
                alpha=0.65,
                s=18,
                label=f'{domain} - {label_name}',
            )

    ax.set_title(title)
    ax.set_xlabel('t-SNE 1')
    ax.set_ylabel('t-SNE 2')
    ax.grid(True, alpha=0.2)
    ax.legend(loc='best', fontsize=9, frameon=True)
    plt.tight_layout()
    plt.show()

    return plot_df

In [ ]:
# t-SNE ANTES da Fase 1
source_sample = sample_df(source_df, n=800, seed=cfg.seed)
rop_sample = sample_df(rop_train_df, n=800, seed=cfg.seed)

model_before = EfficientNetDANN(num_classes=2, pre_trained=True).to(device)

feat_src_before, meta_src_before = extract_features(model_before, source_sample, domain_name='ORIGA', image_size=cfg.image_size, device=device)
feat_rop_before, meta_rop_before = extract_features(model_before, rop_sample, domain_name='ROP', image_size=cfg.image_size, device=device)

features_before = np.vstack([feat_src_before, feat_rop_before])
meta_before = pd.concat([meta_src_before, meta_rop_before], ignore_index=True)
emb_before = run_tsne(features_before, seed=cfg.seed, perplexity=30.0)
plot_before = make_tsne_plot(emb_before, meta_before, 't-SNE — Antes da Fase 1 (EfficientNet-B0 pré-treinado)')

In [ ]:
# t-SNE DEPOIS da Fase 1 (melhor fold)
phase1_dir = Path(cfg.output_dir) / 'checkpoints' / 'phase1'
results_json = phase1_dir / 'phase1_results.json'

if results_json.exists():
    with open(results_json, 'r', encoding='utf-8') as f:
        p1 = json.load(f)
    fold_results = p1.get('fold_results', [])
    best_idx = int(np.argmax([fr['best_val_auc'] for fr in fold_results]))
    best_ckpt = Path(p1['checkpoint_paths'][best_idx])
else:
    best_ckpt = phase1_dir / 'fold1_best.pth'

print('Checkpoint escolhido:', best_ckpt)
assert best_ckpt.exists(), f'Checkpoint não encontrado: {best_ckpt}'

model_after = EfficientNetDANN(num_classes=2, pre_trained=False).to(device)
ckpt = torch.load(best_ckpt, map_location=device)
state_dict = ckpt['model_state_dict'] if isinstance(ckpt, dict) and 'model_state_dict' in ckpt else ckpt
model_after.load_state_dict(state_dict, strict=True)

feat_src_after, meta_src_after = extract_features(model_after, source_sample, domain_name='ORIGA', image_size=cfg.image_size, device=device)
feat_rop_after, meta_rop_after = extract_features(model_after, rop_sample, domain_name='ROP', image_size=cfg.image_size, device=device)

features_after = np.vstack([feat_src_after, feat_rop_after])
meta_after = pd.concat([meta_src_after, meta_rop_after], ignore_index=True)
emb_after = run_tsne(features_after, seed=cfg.seed, perplexity=30.0)
plot_after = make_tsne_plot(emb_after, meta_after, 't-SNE — Depois da Fase 1 (EfficientNet-B0 + DANN)')

In [ ]:
# Salvar pontos 2D para análise externa
tsne_dir = Path(cfg.output_dir) / 'tsne'
tsne_dir.mkdir(parents=True, exist_ok=True)

plot_before.to_csv(tsne_dir / 'tsne_before_phase1.csv', index=False)
plot_after.to_csv(tsne_dir / 'tsne_after_phase1.csv', index=False)

print('Arquivos salvos em:', tsne_dir)
print('-', tsne_dir / 'tsne_before_phase1.csv')
print('-', tsne_dir / 'tsne_after_phase1.csv')